# EEG 3C + 3D — Exact 3B, 24-row epochs, per-subject models

This Colab is intentionally small. The heavy work is in `eeg_3c_epoch_pca.py` and `eeg_3d_subject_models.py`.

- Uses `3B_Exact_40_25_Split`
- 24 rows = 1 epoch/sample before PCA
- target defaults to `load_in_sequence`
- separate PCA + KNN/Logistic/SVM for each subject
- reports MAE + Spearman (and accuracy)
- saves epoch-by-epoch target + predictions
- supports all 6 datasets (development train/validation/holdout + holdout-subject train/validation/holdout)


In [ ]:
!pip -q install pandas numpy scipy scikit-learn matplotlib
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, importlib

CODE_DIR = Path('/content/drive/MyDrive/EEG Shared/Code/Code Luke')
if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

import eeg_3c_epoch_pca
import eeg_3d_subject_models
importlib.reload(eeg_3c_epoch_pca)
importlib.reload(eeg_3d_subject_models)

from eeg_3c_epoch_pca import get_exact_3b_paths, build_all_six_epoch_caches
from eeg_3d_subject_models import (
    DEFAULT_HYPERPARAMETERS, run_development_models,
    run_holdout_subject_models, combine_all_six_epoch_predictions
)
print('3C/3D Python files loaded.')


In [ ]:
SPLIT_FOLDER = Path('/content/drive/MyDrive/EEG Shared/Processed_data/Kosachenko_All_Subjects/3B_Exact_40_25_Split')
OUTPUT_FOLDER = Path('/content/drive/MyDrive/EEG Shared/Results/Luke/3C_3D_Exact3B_Subject_Models')
CACHE_FOLDER = OUTPUT_FOLDER / 'epoch_cache'

# Earlier instruction: predict position/load in the sequence, not old total sequence size.
TARGET_COLUMN = 'load_in_sequence'
EPOCH_ROWS = 24
PC_COUNT = 20
CHUNK_ROWS = 50000
FORCE_REBUILD_CACHE = False

paths = get_exact_3b_paths(SPLIT_FOLDER)
for name, path in paths.items():
    print(name, '->', path, '| exists:', path.exists())


## 3C — flatten every 24-row epoch

This creates one sample per complete `(subject, epoch_uid)` block and keeps each epoch UID so predictions can be returned per epoch.


In [ ]:
datasets, FEATURE_COLUMNS = build_all_six_epoch_caches(
    split_folder=SPLIT_FOLDER,
    cache_folder=CACHE_FOLDER,
    target_column=TARGET_COLUMN,
    epoch_rows=EPOCH_ROWS,
    chunk_rows=CHUNK_ROWS,
    force_rebuild=FORCE_REBUILD_CACHE,
)

for name, data in datasets.items():
    print(name, '| X:', data['X'].shape, '| epochs:', len(data['y']), '| subjects:', len(set(data['subjects'])))


## 3D — development subjects

Fits a separate scaler + PCA + KNN/Logistic/SVM for each of the 40 development subjects. Saves metrics and per-epoch predictions for train, validation, and holdout.


In [ ]:
dev_metrics, dev_predictions, dev_files = run_development_models(
    datasets=datasets,
    output_folder=OUTPUT_FOLDER,
    pc_count=PC_COUNT,
    hyperparameters=DEFAULT_HYPERPARAMETERS,
)

print(dev_files)
display(dev_metrics.head(20))

dev_summary = dev_metrics.groupby(['model','split'])[['mae','spearman','accuracy']].agg(['mean','std']).round(4)
display(dev_summary)


In [ ]:
# Example epoch-by-epoch output
dev_wide = (
    dev_predictions.pivot_table(
        index=['subject_id','subject_group','split','epoch_uid','target'],
        columns='model', values='prediction', aggfunc='first'
    ).reset_index()
    .rename(columns={
        'KNN':'knn_prediction',
        'Logistic Regression':'logistic_prediction',
        'SVM':'svm_prediction'
    })
)
display(dev_wide.head(20))


## Final 25 holdout subjects

Run this only after hyperparameters are fixed. It produces the other 3 datasets and therefore completes epoch-by-epoch predictions for all 6 datasets.


In [ ]:
# FINAL 25 SUBJECTS — run when ready
RUN_FINAL_25 = False

if RUN_FINAL_25:
    final_metrics, final_predictions, final_files = run_holdout_subject_models(
        datasets=datasets,
        output_folder=OUTPUT_FOLDER,
        pc_count=PC_COUNT,
        hyperparameters=DEFAULT_HYPERPARAMETERS,
    )
    print(final_files)
    display(final_metrics.head(20))

    combined_path, all_six_predictions = combine_all_six_epoch_predictions(OUTPUT_FOLDER)
    print('All 6 datasets prediction file:', combined_path)
    display(all_six_predictions.head(20))
else:
    print('RUN_FINAL_25 is False. Development results are complete; final 25 subjects were not touched.')


## Output columns

The wide epoch prediction CSV contains:

`subject_id, subject_group, split, epoch_uid, target, knn_prediction, logistic_prediction, svm_prediction`

The metrics CSV contains MAE, Spearman correlation, and accuracy for every subject/model/split.
